<a href="https://colab.research.google.com/github/Nikoulty/codigo3/blob/main/codex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import collections
import io
import re
import time
import pandas as pd
from google.colab import files

# =====================================================================
# 1. DEFINICIÓN DEL SISTEMA DE GESTIÓN DE PACIENTES
# =====================================================================
class SistemaPacientes:
    def __init__(self):
        self.pacientes = []

    def registrar_paciente(self, id_paciente, nombre, sintomas):
        """Inserta un paciente manteniendo el orden del arreglo por ID (como texto)."""
        nuevo_paciente = {
            "id": str(id_paciente).strip(),
            "nombre": str(nombre).strip(),
            "sintomas": str(sintomas).lower().strip()
        }
        self.pacientes.append(nuevo_paciente)
        self.pacientes.sort(key=lambda x: x["id"])

    def buscar_por_id_binaria(self, id_buscar):
        """Búsqueda Binaria para texto (IDs como P001)."""
        id_buscar = str(id_buscar).strip()
        inicio = 0
        fin = len(self.pacientes) - 1
        pasos = 0
        while inicio <= fin:
            pasos += 1
            medio = (inicio + fin) // 2
            if self.pacientes[medio]["id"] == id_buscar:
                return self.pacientes[medio], pasos
            elif self.pacientes[medio]["id"] < id_buscar:
                inicio = medio + 1
            else:
                fin = medio - 1
        return None, pasos

    def buscar_por_nombre_secuencial(self, nombre_buscar):
        """Búsqueda Secuencial por nombre."""
        resultados = []
        nombre_buscar = nombre_buscar.lower().strip()
        pasos = 0
        for paciente in self.pacientes:
            pasos += 1
            if nombre_buscar in paciente["nombre"].lower():
                resultados.append(paciente)
        return resultados, pasos

    def analizar_patrones_sintomas(self, palabra_clave):
        """Búsqueda de Texto en el historial de enfermedades."""
        coincidencias = []
        palabra_clave = palabra_clave.lower().strip()
        for paciente in self.pacientes:
            if re.search(r'\b' + re.escape(palabra_clave) + r'\b', paciente["sintomas"]):
                coincidencias.append(paciente)
        return coincidencias

    def obtener_sintomas_frecuentes(self, top_n=5):
        """Filtra conectores comunes y extrae los diagnósticos más repetidos."""
        todos_los_sintomas = []
        # Conectores y palabras vacías optimizadas para tu columna "enfermedades"
        conectores = {"y", "o", "con", "el", "la", "los", "del", "un", "una",
                      "presenta", "tiene", "en", "de", "por", "alta", "seca", "congestión", "nan"}
        for paciente in self.pacientes:
            palabras = re.findall(r'\b\w+\b', paciente["sintomas"])
            filtradas = [p for p in palabras if p not in conectores]
            todos_los_sintomas.extend(filtradas)
        return collections.Counter(todos_los_sintomas).most_common(top_n)

    def exportar_informe_txt(self, pasos_id, pasos_nombre, top_sintomas, nombre_archivo="informe_clinico.txt"):
        total_pacientes = len(self.pacientes)

        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write("=====================================================================\n")
            f.write("                  INFORME DE RENDIMIENTO Y CLÍNICO                   \n")
            f.write("=====================================================================\n")
            f.write(f"Volumen de Datos Cargados: {total_pacientes} pacientes.\n\n")

            f.write("1. COMPARATIVA DE EFICIENCIA DE ALGORITMOS (Pasos requeridos):\n")
            f.write("---------------------------------------------------------------------\n")
            f.write(f"Búsqueda Binaria (Por ID):     {pasos_id:<6} pasos\n")
            f.write(f"Búsqueda Secuencial (Nombre):  {pasos_nombre:<6} pasos\n\n")

            f.write("GRÁFICO DE RENDIMIENTO EN TEXTO (Menos bloques '█' es mejor):\n")
            max_pasos = max(pasos_id, pasos_nombre, 1)

            ancho_barra_id = max(1, int((pasos_id / max_pasos) * 30))
            ancho_barra_nombre = max(1, int((pasos_nombre / max_pasos) * 30))

            barra_binaria = "█" * ancho_barra_id
            barra_secuencial = "█" * ancho_barra_nombre

            f.write(f"Búsq. Binaria:    [{barra_binaria:<30}] ({pasos_id} op)\n")
            f.write(f"Búsq. Secuencial: [{barra_secuencial:<30}] ({pasos_nombre} op)\n\n")

            f.write("2. TOP PATOLOGÍAS / ENFERMEDADES DETECTADAS:\n")
            f.write("---------------------------------------------------------------------\n")
            if top_sintomas:
                max_conteo = max([c for _, c in top_sintomas], default=1)
                for r, (sintoma, c) in enumerate(top_sintomas, 1):
                    porcentaje = (c / total_pacientes) * 100
                    barra_sintoma = "█" * int((c / max_conteo) * 25)
                    f.write(f"[{r}] {sintoma.capitalize():<15} : {c:<4} casos ({porcentaje:.1f}%) | {barra_sintoma}\n")
            else:
                f.write("   No hay registros suficientes de enfermedades.\n")

            f.write("\n=====================================================================\n")
            f.write("Informe generado automáticamente por el Sistema de Gestión de Pacientes.\n")
            f.write("=====================================================================\n")

# =====================================================================
# 2. CARGA DIRECTA APUNTANDO A TU EXCEL
# =====================================================================
sys = SistemaPacientes()
NOMBRE_EXCEL = "Lista_Pacientes_100.xlsx"

print(f"🔄 Cargando de forma nativa '{NOMBRE_EXCEL}'...")
print("-----------------------------------------------------------------------------------------")

try:
    df = pd.read_excel(NOMBRE_EXCEL)
    # Limpiamos nombres de columnas
    df.columns = [str(col).strip().lower() for col in df.columns]

    # Vinculación directa con las columnas reales de tu archivo
    col_id = 'id'
    col_nombre = 'nombre'
    col_sintomas = 'enfermedades' # Forzado a tu columna real

    # Validación de seguridad
    if col_id not in df.columns or col_nombre not in df.columns or col_sintomas not in df.columns:
        print("⚠️ Advertencia: Las columnas no tienen el formato exacto. Usando mapeo alternativo...")
        col_id = next((c for c in df.columns if 'id' in c), None)
        col_nombre = next((c for c in df.columns if 'nombre' in c), None)
        col_sintomas = next((c for c in df.columns if 'enfermedad' in c or 'sintoma' in c), None)

    df = df.dropna(subset=[col_id, col_nombre])

    for _, fila in df.iterrows():
        sys.registrar_paciente(
            id_paciente=fila[col_id],
            nombre=fila[col_nombre],
            sintomas=fila[col_sintomas] if pd.notna(fila[col_sintomas]) else "Sin registros"
        )

    print(f"✅ ¡Éxito total! Se cargaron correctamente {len(sys.pacientes)} pacientes desde tu Excel.")

except FileNotFoundError:
    print(f"\n❌ Error: El archivo '{NOMBRE_EXCEL}' no está en la barra lateral.")
    print("Asegúrate de que no se haya borrado o que no tenga un '(1)' en el nombre debido a múltiples descargas.")
except Exception as e:
    print(f"\n❌ Error al procesar el archivo Excel: {e}")

# Valores de control iniciales
ultimos_pasos_id = 1
ultimos_pasos_nombre = len(sys.pacientes)

if len(sys.pacientes) == 0:
    print("\n🛑 No se pudo inicializar la base de datos.")
else:
    # =====================================================================
    # 3. INTERFAZ DE MENÚ INTERACTIVO
    # =====================================================================
    while True:
        print("\n==================================================")
        print("      SISTEMA DE GESTIÓN HOSPITALARIA     ")
        print("==================================================")
        print(f" Pacientes en memoria: {len(sys.pacientes)}")
        print("--------------------------------------------------")
        print("1. Registrar un nuevo paciente (Manual)")
        print("2. Buscar paciente por ID (Búsqueda Binaria)")
        print("3. Buscar paciente por Nombre (Búsqueda Secuencial)")
        print("4. Analizar patrón clínico (Búsqueda de Texto)")
        print("5. GENERAR Y DESCARGAR INFORME TEXTUAL (.TXT)")
        print("6. Salir del sistema")
        print("==================================================")

        opcion = input("Seleccione una opción (1-6): ").strip()
        print("\n")

        if opcion == "1":
            nuevo_id = input("Ingrese ID único (ej: P101): ").strip()
            if nuevo_id:
                paciente_existe, _ = sys.buscar_por_id_binaria(nuevo_id)
                if paciente_existe:
                    print("❌ Error: Este ID ya existe.")
                    continue
                nuevo_nombre = input("Ingrese nombre completo: ").strip()
                nuevos_sintomas = input("Registre las enfermedades/diagnósticos: ").strip()
                if nuevo_nombre and nuevos_sintomas:
                    sys.registrar_paciente(nuevo_id, nuevo_nombre, nuevos_sintomas)
                    print(f"✅ Paciente {nuevo_nombre} registrado correctamente.")

        elif opcion == "2":
            id_buscar = input("Ingrese el ID a buscar (ej: P001): ").strip()
            if id_buscar:
                paciente, pasos = sys.buscar_por_id_binaria(id_buscar)
                ultimos_pasos_id = pasos
                if paciente:
                    print(f"📍 [Código Encontrado] Pasos ejecutados por la Búsqueda Binaria: {pasos}")
                    print(f"   • Paciente: {paciente['nombre']}")
                    print(f"   • Historial Clínico: {paciente['sintomas'].capitalize()}")
                else:
                    print("❌ No se encontró ningún paciente con ese identificador.")

        elif opcion == "3":
            nombre_buscar = input("Ingrese el nombre a buscar: ").strip()
            if nombre_buscar:
                resultados, pasos = sys.buscar_por_nombre_secuencial(nombre_buscar)
                ultimos_pasos_nombre = pasos
                print(f"📋 Coincidencias encontradas: {len(resultados)} | Pasos de la Búsqueda Secuencial: {pasos}")
                for p in resultados[:5]:
                    print(f"   • [ID: {p['id']}] {p['nombre']}")

        elif opcion == "4":
            sintoma_buscar = input("Ingrese la enfermedad clave a rastrear (ej: Asma, Diabetes): ").strip()
            if sintoma_buscar:
                coincidencias = sys.analizar_patrones_sintomas(sintoma_buscar)
                print(f"📊 Casos clínicos detectados con esa patología: {len(coincidencias)}")
                for p in coincidencias[:5]:
                    print(f"   • {p['nombre']} (ID: {p['id']}): {p['sintomas'].capitalize()}")

        elif opcion == "5":
            print("--- GENERANDO REPORTE DIGITAL EN TEXTO PLANO ---")
            top_sintomas = sys.obtener_sintomas_frecuentes(5)
            sys.exportar_informe_txt(ultimos_pasos_id, ultimos_pasos_nombre, top_sintomas)
            print("\n⬇️ Descargando 'informe_clinico.txt'...")
            try:
                files.download('informe_clinico.txt')
                print("✅ Descarga exitosa. Revisa tu carpeta local de descargas.")
            except Exception as e:
                print("❌ Descarga bloqueada por el navegador. Puedes bajarlo manualmente abriendo el panel de archivos de la izquierda.")

        elif opcion == "6":
            print("👋 Saliendo del sistema clínico hospitalario. ¡Buen día!")
            break

🔄 Cargando de forma nativa 'Lista_Pacientes_100.xlsx'...
-----------------------------------------------------------------------------------------
⚠️ Advertencia: Las columnas no tienen el formato exacto. Usando mapeo alternativo...
✅ ¡Éxito total! Se cargaron correctamente 100 pacientes desde tu Excel.

      SISTEMA DE GESTIÓN HOSPITALARIA     
 Pacientes en memoria: 100
--------------------------------------------------
1. Registrar un nuevo paciente (Manual)
2. Buscar paciente por ID (Búsqueda Binaria)
3. Buscar paciente por Nombre (Búsqueda Secuencial)
4. Analizar patrón clínico (Búsqueda de Texto)
5. GENERAR Y DESCARGAR INFORME TEXTUAL (.TXT)
6. Salir del sistema
